In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Detect project directory
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_dir = current_dir.parent
else:
    project_dir = current_dir

video_path = project_dir / "data" / "synthetic_videos" / "synthetic_circle.mp4"

output_video_path = project_dir / "outputs" / "processed_videos" / "frame_difference_synthetic.mp4"
output_plot_path = project_dir / "outputs" / "plots" / "trajectory_synthetic.png"
output_csv_path = project_dir / "outputs" / "plots" / "trajectory_synthetic.csv"

output_video_path.parent.mkdir(parents=True, exist_ok=True)
output_plot_path.parent.mkdir(parents=True, exist_ok=True)

print("Project directory:", project_dir)
print("Video path:", video_path)
print("Video exists:", video_path.exists())

In [ ]:
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("Could not open the video")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Frame count:", frame_count)
print("Resolution:", width, "x", height)

ret, frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Could not read the first frame")

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 5))
plt.imshow(frame_rgb)
plt.axis("off")
plt.title("First frame")
plt.show()

In [ ]:
def track_motion_with_frame_difference(
    input_video_path: Path,
    output_video_path: Path,
    threshold_value: int = 25,
    min_area: int = 300,
    kernel_size: int = 5,
):
    """
    Track the main moving object using frame difference.

    The method compares consecutive grayscale frames, thresholds the difference,
    extracts contours, and follows the centroid of the largest moving region.
    """

    cap = cv2.VideoCapture(str(input_video_path))

    if not cap.isOpened():
        raise RuntimeError("Could not open the input video")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

    ret, previous_frame = cap.read()

    if not ret:
        raise RuntimeError("Could not read the first frame")

    previous_gray = cv2.cvtColor(previous_frame, cv2.COLOR_BGR2GRAY)
    previous_gray = cv2.GaussianBlur(previous_gray, (5, 5), 0)

    kernel = np.ones((kernel_size, kernel_size), np.uint8)

    trajectory = []
    frame_index = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        # Absolute difference between consecutive frames
        frame_difference = cv2.absdiff(previous_gray, gray)

        # Threshold to isolate moving regions
        _, motion_mask = cv2.threshold(
            frame_difference,
            threshold_value,
            255,
            cv2.THRESH_BINARY
        )

        # Morphological operations to reduce noise
        motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel)
        motion_mask = cv2.dilate(motion_mask, kernel, iterations=2)

        # Find contours of moving regions
        contours, _ = cv2.findContours(
            motion_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        detected = False
        cx, cy = None, None

        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(largest_contour)

            if area > min_area:
                x, y, w, h = cv2.boundingRect(largest_contour)

                cx = x + w // 2
                cy = y + h // 2

                trajectory.append({
                    "frame": frame_index,
                    "x": cx,
                    "y": cy,
                    "area": area
                })

                detected = True

                # Draw bounding box
                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

                # Draw centroid
                cv2.circle(
                    frame,
                    (cx, cy),
                    5,
                    (0, 0, 255),
                    -1
                )

        # Draw trajectory
        trajectory_points = [(point["x"], point["y"]) for point in trajectory]

        for i in range(1, len(trajectory_points)):
            cv2.line(
                frame,
                trajectory_points[i - 1],
                trajectory_points[i],
                (255, 0, 0),
                2
            )

        # Add text information
        status_text = "Detected" if detected else "Not detected"
        cv2.putText(
            frame,
            f"Frame: {frame_index} | Motion: {status_text}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        if detected:
            cv2.putText(
                frame,
                f"Centroid: ({cx}, {cy})",
                (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

        writer.write(frame)

        previous_gray = gray.copy()
        frame_index += 1

    cap.release()
    writer.release()

    trajectory_df = pd.DataFrame(trajectory)

    return trajectory_df

In [ ]:
trajectory_df = track_motion_with_frame_difference(
    input_video_path=video_path,
    output_video_path=output_video_path,
    threshold_value=25,
    min_area=300,
    kernel_size=5
)

print("Processed video saved at:", output_video_path)
print("Number of detected points:", len(trajectory_df))

trajectory_df.head()

In [ ]:
trajectory_df.to_csv(output_csv_path, index=False)

plt.figure(figsize=(8, 5))
plt.plot(trajectory_df["x"], trajectory_df["y"], marker="o", markersize=2)
plt.gca().invert_yaxis()
plt.xlabel("X position [pixels]")
plt.ylabel("Y position [pixels]")
plt.title("Estimated trajectory using frame difference")
plt.grid(True)
plt.savefig(output_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Trajectory CSV saved at:", output_csv_path)
print("Trajectory plot saved at:", output_plot_path)